# Usar los modelos finales con tus propios compuestos

**Para quién:** la patrocinadora y quien herede el proyecto. **Qué hace:** recibe una lista de SMILES y devuelve, para cada compuesto, (1) la probabilidad de que cruce la barrera hematoencefálica, (2) la potencia que el modelo de la fase 2 predice contra cada una de las 26 dianas, y (3) **qué tan cerca queda el compuesto de lo que los modelos vieron al entrenarse**. No hay servicio ni punto de acceso: la reunión del 3 de noviembre decidió que lo que se entrega es este cuaderno.

**Qué no hace, a propósito.** No decide un corte: la probabilidad de la fase 1 se entrega tal cual, porque cortarla en 0.5 cambia de sesgo cuando el compuesto es distinto de lo entrenado ([1.2](../docs/decisiones.md#d1-2)); y no combina las dos fases en un ranking, porque esa combinación sigue pendiente ([C.3](../docs/decisiones.md#dc-3)). Un BBB− **no descarta** un compuesto.

## 1. Qué respalda estas cifras

Los modelos se entrenaron con las filas de **ajuste** (entrenamiento y validación), sin la prueba reservada. Las cifras de desempeño **no se recalculan aquí**: son las de las corridas registradas, y la diferencia entre la validación cruzada y la prueba es la advertencia más importante de este cuaderno.

| fase | modelo | validación cruzada agrupada | prueba reservada (química más nueva) |
|:--|:--|:--|:--|
| 1 · permeabilidad | `svm_rbf` | MCC 0.7017, AUROC 0.919 | MCC 0.339, AUROC 0.793 (1,499 compuestos) |
| 2 · potencia | `aft` censurado | concordancia 0.761 (0.632 en pares que solo tiene NPASS) | concordancia 0.720 (**0.530 en NPASS, 117 pares**) |

**Una cifra de validación cruzada no describe a la química nueva**: con el mismo modelo, exigir que ningún compuesto de entrenamiento se parezca más de 0.4 a uno de validación baja el MCC de la fase 1 en 33 puntos. Por eso el cuaderno calcula la similitud de cada consulta con el entrenamiento y avisa cuando es baja (sección 4). Es una **heurística, no una garantía**: la similitud al vecino más cercano no anticipa bien la dificultad de la prueba reservada.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd

from src import inferencia

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

modelos = inferencia.cargar()
print(f"Fase 1: {modelos.meta['filas']['fase1']:,} compuestos · {len(modelos.columnas_fase1)} descriptores")
print(f"Fase 2: {modelos.meta['filas']['fase2']:,} pares · {len(modelos.columnas_fase2)} descriptores + {len(modelos.dianas)} dianas")

Fase 1: 6,306 compuestos · 151 descriptores
Fase 2: 56,241 pares · 201 descriptores + 26 dianas


## 2. Tus compuestos

Edita este diccionario: nombre → SMILES. Los de abajo son ejemplos conocidos, más un SMILES inválido y una molécula diminuta para ver cómo se comportan los avisos.

In [2]:
compuestos = {
    "cafeína": "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "nicotina": "CN1CCC[C@H]1c1cccnc1",
    "galantamina": "COc1ccc2c3c1O[C@H]1C[C@@H](O)C=C[C@@]31CCN(C)C2",
    "resveratrol": "Oc1ccc(/C=C/c2cc(O)cc(O)c2)cc1",
    "curcumina": "COc1cc(/C=C/C(=O)CC(=O)/C=C/c2ccc(O)c(OC)c2)ccc1O",
    "quercetina": "O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12",
    "metano (diminuto)": "C",
    "no es un SMILES": "esto no es un smiles",
}

## 3. Predicciones

Una fila por compuesto. Un SMILES que no se puede usar sale con `NaN` y su motivo, y no afecta a los demás.

In [3]:
resultado = inferencia.predecir(list(compuestos.values()), modelos)
resultado.insert(0, "nombre", list(compuestos))
resultado[["nombre", "p_bbb_mas", "mejor_diana", "pActividad_max", "similitud_fase1", "similitud_fase2",
           "aviso_extrapolacion", "motivo"]].round(3)

,nombre,p_bbb_mas,mejor_diana,pActividad_max,similitud_fase1,similitud_fase2,aviso_extrapolacion,motivo
0,cafeína,0.944,IL1B,5.247,1.000,1.000,False,
1,nicotina,0.933,MAPT,4.928,1.000,1.000,False,
2,galantamina,0.962,DRD2,6.365,1.000,1.000,False,
3,resveratrol,0.182,APP,6.303,0.379,1.000,True,
4,curcumina,0.184,TSPO,6.878,0.439,1.000,False,
5,quercetina,0.023,MAOB,5.335,0.526,1.000,False,
6,metano (diminuto),0.914,IL6,4.919,1.000,0.167,True,
7,no es un SMILES,NaN,NaN,NaN,NaN,NaN,False,SMILES ilegible


**Cómo leerla.**

- `p_bbb_mas`: probabilidad de cruzar la barrera, de 0 a 1. **No es una clase**: 0.5 no es un límite que el modelo garantice.
- `pActividad_<diana>`: potencia predicha (−log₁₀ de la concentración en M; 5 ≈ 10 µM, 6 ≈ 1 µM, 7 ≈ 0.1 µM). **El modelo predice una potencia para cada diana, también las que el compuesto nunca ha tocado**: es una extrapolación de lo aprendido sobre pares medidos, no una medición. `mejor_diana` y `pActividad_max` solo ordenan esas predicciones.
- `similitud_fase1` y `similitud_fase2`: Tanimoto (huellas de Morgan, radio 2) con el compuesto de entrenamiento más parecido. 1.0 significa que **el compuesto estaba en el entrenamiento**, y entonces su predicción no es una prueba del modelo.
- `aviso_extrapolacion`: `True` si alguna similitud es menor de 0.4. Un `False` no garantiza que la predicción sea buena.

## 4. Qué tan cerca queda cada compuesto de lo ya visto

In [4]:
cerca = resultado[["nombre", "similitud_fase1", "similitud_fase2", "aviso_extrapolacion"]].round(3)
cerca.sort_values("similitud_fase1")

,nombre,similitud_fase1,similitud_fase2,aviso_extrapolacion
3,resveratrol,0.379,1.000,True
4,curcumina,0.439,1.000,False
5,quercetina,0.526,1.000,False
0,cafeína,1.000,1.000,False
1,nicotina,1.000,1.000,False
2,galantamina,1.000,1.000,False
6,metano (diminuto),1.000,0.167,True
7,no es un SMILES,NaN,NaN,False


## 5. La potencia contra cada diana

Las diez dianas con mayor potencia predicha para un compuesto, para ver el reparto y no solo el máximo.

In [5]:
def por_diana(nombre: str, n: int = 10) -> pd.Series:
    fila = resultado.loc[resultado["nombre"] == nombre].iloc[0]
    columnas = [f"pActividad_{d}" for d in modelos.dianas]
    s = fila[columnas].astype(float)
    s.index = [c.removeprefix("pActividad_") for c in columnas]
    return s.sort_values(ascending=False).head(n).round(2)


por_diana("galantamina")

DRD2      6.37
GRIN2B    6.22
TNF       6.06
TSPO      5.90
APP       5.86
LRRK2     5.78
IL6       5.63
NFE2L2    5.53
MAPT      5.35
COMT      5.32
Name: 2, dtype: float64

## 6. Cómo reentrenar o actualizar

Los modelos viven en `models/finales/` (unos 3 MB) y se regeneran con un comando, que usa las filas de ajuste y **no toca la prueba reservada**:

```bash
./env312/bin/python scripts/entrenar_finales.py
```

El script registra la corrida (parámetros, commit, `sha256` de los datos) en `experimentos/`. Si llegan datos nuevos, **hay que reevaluar con la escalera de semejanza** ([1.2](../docs/decisiones.md#d1-2)) y no con una partición al azar, que infla la cifra. Con otra versión de las librerías las predicciones pueden cambiar en el tercer decimal; `tests/test_inferencia.py` lo detecta contra 25 predicciones de referencia.

**Uso no comercial:** una de las fuentes de la fase 2, NPASS, es CC BY-NC.

# Declaración de uso de IA y contribución del equipo

### Uso de herramientas de IA

| Herramienta utilizada | Propósito de uso | Alcance del apoyo (secciones) | Responsabilidad: qué revisó o validó el equipo |
|---|---|---|---|
| Anthropic. (2026). Claude Code [Agente de programación]. https://claude.com/claude-code | Escribió el módulo `src/inferencia.py`, el script `scripts/entrenar_finales.py`, `tests/test_inferencia.py` y este cuaderno | Todas las secciones. Las dos columnas derivadas de la fase 1 (`regla_2p` y `tpsa_por_mw`) las tomó de `scripts/experimento_derivadas.py` | **Lo que el equipo decidió:** entregar un cuaderno y no un servicio (reunión del 3 de noviembre); entrenar con las filas de ajuste y no con todas, aun renunciando a unos +0.007 de concordancia en la fase 2 (4 de noviembre); y no fijar un corte de probabilidad ni combinar las dos fases. **Lo que se comprobó:** que los descriptores calculados desde un SMILES coinciden bit a bit con los de entrenamiento en 150 moléculas de cada fase, y que el modelo recargado reproduce al entrenado (diferencia 0.0) |

### Decisiones principales

1. **Cuaderno, no endpoint.** *Alternativa descartada:* un servicio, porque la patrocinadora pidió lo contrario y no hay quien lo opere.
2. **Sin corte.** *Alternativa descartada:* devolver «BBB+/BBB−» a 0.5, que bajo un cambio de distribución cambia de sesgo.
3. **Aviso de extrapolación por similitud.** *Alternativa descartada:* no avisar. Se asume que es una heurística imperfecta y se dice.